# PyTorch interview coding lab

Install a compatible PyTorch build into the notebook kernel environment if missing. Exercises use CPU tensors; no CUDA device or model download is needed.

Use only the synthetic fixtures supplied here. No private data is included. This notebook never installs packages automatically. Replace each `...`, run the exercise cell, then its separate test cell. Failures before completion are intentional. Tests cover selected cases, not every production input. Reference solutions are text at the end and are not executed automatically.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('Exercise device: CPU; no CUDA execution required')


## 01 / TENSORS & SHAPES: Split tokens into attention heads

Transform [batch, tokens, width] into [batch, heads, tokens, head_width] without scrambling token order.

### Hints
- **Reshape into [B, T, H, D]:** Keep batch and token dimensions first, then split width into heads and per-head width.
- **Move heads before tokens:** Swap axes 1 and 2. A transpose is often a non-contiguous view.

### Expected behavior
For x shaped [2, 4, 12] and heads=3: output shape [2, 3, 4, 4]. Head slices preserve the original token's contiguous width groups.


In [ ]:
import torch

def split_heads(x: torch.Tensor, heads: int) -> torch.Tensor:
    batch, tokens, width = x.shape
    if heads <= 0 or width % heads:
        raise ValueError("heads must divide width")
    separated = ...
    return ...


In [ ]:
x = torch.arange(96.0).reshape(2, 4, 12)
y = split_heads(x, 3)
assert y.shape == (2, 3, 4, 4)
assert torch.equal(y[0, 1, 2], x[0, 2, 4:8])
restored = y.transpose(1, 2).reshape(2, 4, 12)
assert torch.equal(restored, x)
try:
    split_heads(x, 5)
except ValueError:
    pass
else:
    raise AssertionError("Invalid head count must be rejected")
print("PASS: head shape, token ordering, reconstruction, invalid heads")


## 02 / NUMERICAL STABILITY: Write numerically stable softmax

Normalize the last dimension without overflow, preserving batch dimensions and gradient flow.

### Hints
- **Subtract the row maximum:** A per-row maximum with keepdim=True broadcasts over the original last axis.
- **Exponentiate shifted logits:** Subtracting a constant leaves softmax unchanged but prevents exp(large positive number).
- **Sum weights along the last axis:** Do not sum across the batch. Preserve the final axis for broadcasting.

### Expected behavior
For [1000, 1001, 1002], probabilities are approximately [0.0900, 0.2447, 0.6652], all finite, with row sum 1.


In [ ]:
import torch

def stable_probs(logits: torch.Tensor) -> torch.Tensor:
    shifted = ...
    weights = ...
    return weights / ...


In [ ]:
logits = torch.tensor([[1000., 1001., 1002.], [3., 1., -2.]], requires_grad=True)
probs = stable_probs(logits)
assert torch.isfinite(probs).all()
assert torch.allclose(probs.sum(dim=-1), torch.ones(2))
assert torch.allclose(probs, torch.softmax(logits, dim=-1), atol=1e-6)
probs[:, 0].sum().backward()
assert logits.grad is not None and torch.isfinite(logits.grad).all()
print("PASS: large logits, normalization, reference parity, gradients")


## 03 / PYTORCH SDPA: Make future tokens invisible

Create a boolean causal mask with the correct SDPA semantics and explicitly disable functional dropout.

### Hints
- **Construct the allowed-position mask:** A lower triangular boolean matrix includes current and previous keys. In SDPA, True means allowed.
- **Disable SDPA dropout for this inference exercise:** Functional SDPA applies the provided dropout probability regardless of an enclosing module's eval setting.

### Expected behavior
Output has the same [B,H,T,D] shape as Q. Perturbing K and V at the final position must not change earlier outputs.


In [ ]:
import torch
from torch.nn import functional as F

def causal_attention(q: torch.Tensor, k: torch.Tensor,
                     v: torch.Tensor) -> torch.Tensor:
    tokens = q.size(-2)
    if k.size(-2) != tokens or v.size(-2) != tokens:
        raise ValueError("This exercise requires equal sequence lengths")
    allowed = ...
    return F.scaled_dot_product_attention(
        q, k, v, attn_mask=allowed, dropout_p=...
    )


In [ ]:
torch.manual_seed(7)
q, k, v = [torch.randn(2, 2, 4, 8) for _ in range(3)]
out = causal_attention(q, k, v)
assert out.shape == q.shape
changed_k, changed_v = k.clone(), v.clone()
changed_k[..., -1, :] += 100
changed_v[..., -1, :] -= 100
changed = causal_attention(q, changed_k, changed_v)
assert torch.allclose(out[..., :-1, :], changed[..., :-1, :], atol=1e-5)
mask = torch.ones(4, 4, dtype=torch.bool).tril()
scores = (q @ k.transpose(-2, -1)) / (q.size(-1) ** 0.5)
reference = scores.masked_fill(~mask, float('-inf')).softmax(-1) @ v
assert torch.allclose(out, reference, atol=1e-5)
print("PASS: output shape, future-token isolation, explicit attention parity")


## 04 / CAUSAL LANGUAGE-MODEL LOSS: Align next-token targets and mask padding

Write a custom next-token loss that shifts once, ignores -100 labels, and rejects an all-ignored batch.

### Hints
- **Keep predictions before the final position:** The prediction at position t supervises the token at position t+1.
- **Keep targets after the first position:** Pair logits at 0..T-2 with labels at 1..T-1.
- **Specify the ignored label:** Cross entropy must use the same ignore index as your label mask.

### Expected behavior
With zero logits over a 3-token vocabulary, loss is log(3) ≈ 1.0986 over valid targets. Masked targets and unused final-position logits have zero gradient.


In [ ]:
import torch
from torch.nn import functional as F

def shifted_lm_loss(logits: torch.Tensor,
                    labels: torch.Tensor) -> torch.Tensor:
    shifted_logits = ...
    shifted_labels = ...
    if not (shifted_labels != -100).any():
        raise ValueError("At least one supervised target is required")
    return F.cross_entropy(
        shifted_logits.reshape(-1, shifted_logits.size(-1)),
        shifted_labels.reshape(-1), ignore_index=...
    )


In [ ]:
import math
logits = torch.zeros(1, 4, 3, requires_grad=True)
labels = torch.tensor([[0, 1, -100, 2]])
loss = shifted_lm_loss(logits, labels)
assert abs(loss.item() - math.log(3)) < 1e-6
loss.backward()
assert torch.equal(logits.grad[:, 1, :], torch.zeros(1, 3))
assert torch.equal(logits.grad[:, -1, :], torch.zeros(1, 3))
assert logits.grad[:, 0, :].abs().sum() > 0
try:
    shifted_lm_loss(logits.detach(), torch.full_like(labels, -100))
except ValueError:
    pass
else:
    raise AssertionError("All-ignored labels must be rejected")
print("PASS: shifted loss, ignored gradients, last-position gradient, empty supervision")


## 05 / AUTOGRAD & OPTIMIZATION: Wire a clean optimizer update

Clear stale gradients, backpropagate, and update parameters exactly once per training step.

### Hints
- **Clear previous gradients:** backward accumulates. This exercise is one update per batch, not intentional gradient accumulation.
- **Compute gradients:** Backward traverses the autograd graph, filling parameter gradients.
- **Apply the optimizer update:** Computing gradients alone does not update weights.

### Expected behavior
For y = w*x, initial w=0, x=1, target=2 and SGD lr=0.1: first w=0.4; after a second clean update w=0.72.


In [ ]:
import torch
from torch import nn

def train_once(model: nn.Module, optimizer: torch.optim.Optimizer,
               x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    model.train()
    ...
    loss = (model(x) - target).square().mean()
    ...
    ...
    return loss.detach()


In [ ]:
model = nn.Linear(1, 1, bias=False)
with torch.no_grad():
    model.weight.zero_()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
x, target = torch.tensor([[1.]]), torch.tensor([[2.]])
first_loss = train_once(model, optimizer, x, target)
assert abs(first_loss.item() - 4.0) < 1e-6
assert abs(model.weight.item() - 0.4) < 1e-6
assert not first_loss.requires_grad
train_once(model, optimizer, x, target)
assert abs(model.weight.item() - 0.72) < 1e-6
print("PASS: backward, parameter update, gradient clearing, detached logging loss")


## 06 / INFERENCE CORRECTNESS: Separate eval mode from autograd

Disable training-only module behavior and gradient recording independently.

### Hints
- **Switch module behavior:** Dropout changes behavior with the module's training flag, not just with gradient tracking.
- **Enter a no-gradient context:** eval alone does not disable autograd. This prediction is not being reused for training.

### Expected behavior
Repeated predictions are identical with dropout disabled, probabilities sum to one, and returned tensors do not require gradients.


In [ ]:
import torch
from torch import nn

def predict(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    ...
    with ...:
        return torch.softmax(model(x), dim=-1)


In [ ]:
torch.manual_seed(11)
model = nn.Sequential(nn.Linear(4, 3), nn.Dropout(p=0.9))
model.train()
x = torch.randn(2, 4, requires_grad=True)
a, b = predict(model, x), predict(model, x)
assert not model.training
assert not a.requires_grad
assert torch.equal(a, b)
assert torch.allclose(a.sum(-1), torch.ones(2))
assert x.grad is None
print("PASS: eval behavior, no autograd, deterministic dropout, normalized output")


# Reference solutions

Try the exercises first. Equivalent implementations can be valid even when the browser's limited reference matcher does not accept them.


## Split tokens into attention heads

```python
import torch

def split_heads(x: torch.Tensor, heads: int) -> torch.Tensor:
    batch, tokens, width = x.shape
    if heads <= 0 or width % heads:
        raise ValueError("heads must divide width")
    separated = x.reshape(batch, tokens, heads, width // heads)
    return separated.transpose(1, 2)
```

Reshaping alone to [B,H,T,D] would group the underlying elements incorrectly. First split the final width axis, then permute axes. reshape also supports inputs whose current strides prevent view.


## Write numerically stable softmax

```python
import torch

def stable_probs(logits: torch.Tensor) -> torch.Tensor:
    shifted = logits - logits.amax(dim=-1, keepdim=True)
    weights = torch.exp(shifted)
    return weights / weights.sum(dim=-1, keepdim=True)
```

Subtracting a shared row constant cancels in softmax's numerator and denominator. Reducing the last axis independently keeps separate batch items from influencing one another. All-masked rows need a separate policy; this exercise assumes finite, nonempty logits.


## Make future tokens invisible

```python
import torch
from torch.nn import functional as F

def causal_attention(q: torch.Tensor, k: torch.Tensor,
                     v: torch.Tensor) -> torch.Tensor:
    tokens = q.size(-2)
    if k.size(-2) != tokens or v.size(-2) != tokens:
        raise ValueError("This exercise requires equal sequence lengths")
    allowed = torch.ones(tokens, tokens, dtype=torch.bool, device=q.device).tril()
    return F.scaled_dot_product_attention(
        q, k, v, attn_mask=allowed, dropout_p=0.0
    )
```

Mask scores before normalization. This exercise is square attention with no padding or KV cache; cached unequal-length attention needs explicit offset handling. We do not pass is_causal together with this explicit mask.


## Align next-token targets and mask padding

```python
import torch
from torch.nn import functional as F

def shifted_lm_loss(logits: torch.Tensor,
                    labels: torch.Tensor) -> torch.Tensor:
    shifted_logits = logits[:, :-1, :]
    shifted_labels = labels[:, 1:]
    if not (shifted_labels != -100).any():
        raise ValueError("At least one supervised target is required")
    return F.cross_entropy(
        shifted_logits.reshape(-1, shifted_logits.size(-1)),
        shifted_labels.reshape(-1), ignore_index=-100
    )
```

This is a custom loss, so it performs its own shift. Many Hugging Face causal-LM classes already shift when labels are supplied—do not apply both. The loss is normalized by valid targets, not all padded positions.


## Wire a clean optimizer update

```python
import torch
from torch import nn

def train_once(model: nn.Module, optimizer: torch.optim.Optimizer,
               x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = (model(x) - target).square().mean()
    loss.backward()
    optimizer.step()
    return loss.detach()
```

The first loss is (0−2)^2=4, giving gradient −4 and update +0.4. The next gradient is −3.2, not −7.2: stale gradients must be cleared. A detached returned loss avoids retaining a training graph in logging history.


## Separate eval mode from autograd

```python
import torch
from torch import nn

def predict(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    model.eval()
    with torch.inference_mode():
        return torch.softmax(model(x), dim=-1)
```

Module mode and autograd mode are separate controls. inference_mode removes extra bookkeeping and imposes additional restrictions; no_grad is also valid for this exercise. Restore train mode before training again. SDPA functional dropout is a separate explicit argument.
